# 00 · Set up your workspace (run this first)

**R for All of Us Bootcamp · before Day 1**

This notebook uses the **Python** kernel. It finds your project, buckets and the All of Us dataset (CDR v9, `C2025Q4R6`), then writes an R file that every bootcamp notebook loads.

1. Open this notebook in JupyterLab and choose the **Python 3** kernel.
2. Run every cell from top to bottom (**Run > Run All Cells**).
3. Check that the last cell prints a value for each variable and no `NOT_FOUND`.
4. Open the Session 1 notebook (R kernel).

Run it again if you create a new workspace or restart from a fresh environment.

### Step 1. Find your Google Cloud project

Reads your workspace's project ID with the `wb` command-line tool.

In [ ]:
import os

GOOGLE_CLOUD_PROJECT = !wb workspace describe --format=json | jq -r .'googleProjectId'
GOOGLE_CLOUD_PROJECT

### Step 2. Make sure your workspace buckets exist

Creates the workspace bucket and a temporary bucket (files there are deleted after 14 days) if they don't exist yet.

In [ ]:
!wb resource resolve --name workspace-bucket || wb resource create gcs-bucket \
    --name=workspace-bucket \
    --cloning=COPY_NOTHING \
    --description="Primary workspace bucket for storing files"


!wb resource resolve --name temporary-workspace-bucket || wb resource create gcs-bucket \
    --name=temporary-workspace-bucket \
    --cloning=COPY_NOTHING \
    --auto-delete=14 \
    --description="Bucket for temporary storage of file data. Send outputs here for automatic cleanup after two weeks."

### Step 3. Save the workspace bucket address

In [ ]:
import os

WORKSPACE_BUCKET_URL = !wb resource resolve --id workspace-bucket
WORKSPACE_BUCKET_URL = WORKSPACE_BUCKET_URL[0]
WORKSPACE_BUCKET_URL

%env WORKSPACE_BUCKET = $WORKSPACE_BUCKET_URL

### Step 4. Save the temporary bucket address

In [ ]:
TEMP_WORKSPACE_BUCKET_URL = !wb resource resolve --id temporary-workspace-bucket
TEMP_WORKSPACE_BUCKET_URL = TEMP_WORKSPACE_BUCKET_URL[0]
TEMP_WORKSPACE_BUCKET_URL

%env TEMP_WORKSPACE_BUCKET = $TEMP_WORKSPACE_BUCKET_URL

### Step 5. List your buckets and set the CDR (C2025Q4R6)

In [ ]:
import json
import subprocess

# Get all resources
resources_json = subprocess.run(
    ["wb", "resource", "list", "--format=json"],
    capture_output=True,
    text=True,
    check=True
).stdout
resources = json.loads(resources_json)

# Extract ALL GCS_BUCKET resources
all_buckets = [r for r in resources if r["resourceType"] == "GCS_BUCKET"]

print(f"Found {len(all_buckets)} GCS buckets:")
for i, bucket in enumerate(all_buckets, 1):
    print(f"{i}. ID: {bucket['id']}, Name: {bucket['bucketName']}")
    
# Or as gs:// URLs
gs_urls = [f"gs://{b['bucketName']}" for b in all_buckets]
print(f"\nGS URLs: {gs_urls}")

dataset_id='C2025Q4R6'

project_id = os.environ.get('GOOGLE_CLOUD_PROJECT')
project_id

WORKSPACE_CDR = f"{project_id}.{dataset_id}"
WORKSPACE_CDR

# Set environment variable
os.environ['WORKSPACE_CDR'] = WORKSPACE_CDR

### Step 6. Find the All of Us dataset attached to your workspace

In [ ]:
import re
import json
import subprocess
import os

resources_json = subprocess.run(
    ["wb", "resource", "list", "--format=json"],
    capture_output=True,
    text=True,
    check=True
).stdout

resources = json.loads(resources_json)

for r in resources:
    if r["resourceType"] in ["BQ_DATASET", "BIGQUERY_DATASET"]:
        dataset = r["datasetId"]

        if re.match(r"^C\d{4}Q\d+R\d+$", dataset):
            os.environ["WORKSPACE_CDR"] = f"{r['projectId']}.{dataset}"
            break

!wb resource resolve --name dataset_test2 || wb resource create bq-dataset \
    --name=dataset_test2 \
    --cloning COPY_REFERENCE \
    --description="bigquery dataset test COPY_REFERENCE"

# %run /home/jupyter/Setting_Env_Variables.ipynb

### Step 7. Collect all variables and save them to `~/.bashrc`

In [ ]:
import json
import subprocess
import os
import re

# --- Extract variables directly into os.environ ---
workspace = json.loads(subprocess.run(
    ["wb", "workspace", "describe", "--format=json"],
    capture_output=True, text=True, check=True
).stdout)

os.environ["GOOGLE_CLOUD_PROJECT"] = workspace["googleProjectId"]

resources = json.loads(subprocess.run(
    ["wb", "resource", "list", "--format=json"],
    capture_output=True, text=True, check=True
).stdout)

os.environ["WORKSPACE_CDR"] = ""

# --- Step 3: Extract workspace resources ---
for r in resources:

    # 1. BUCKET LOGIC
    if r["resourceType"] == "GCS_BUCKET":
        print(f"Found bucket: id={r['id']}, bucketName={r['bucketName']}")

        if "temporary-workspace-bucket" in r["id"]:
            os.environ["WORKSPACE_TEMP_BUCKET"] = f"gs://{r['bucketName']}"

        elif "workspace-bucket" in r["id"]:
            os.environ["WORKSPACE_BUCKET"] = f"gs://{r['bucketName']}"

    # 2. BQ DATASET LOGIC  ← SAME LEVEL as bucket logic
    elif r["resourceType"] in ["BQ_DATASET", "BIGQUERY_DATASET"]:

        if os.environ.get("WORKSPACE_CDR", "") == "":

            dataset_id = r["datasetId"]

            # Match real All of Us CDR pattern like C2025Q4R6
            if re.match(r"^C\d{4}Q\d+R\d+$", dataset_id):

                os.environ["WORKSPACE_CDR"] = f"{r['projectId']}.{dataset_id}"

                print(f"Successfully set WORKSPACE_CDR to: {os.environ['WORKSPACE_CDR']}")

# --- Print what we got ---
print("\nVariables extracted:")
for var in ["GOOGLE_CLOUD_PROJECT", "WORKSPACE_BUCKET", 
            "WORKSPACE_TEMP_BUCKET", "WORKSPACE_CDR"]:
    value = os.environ.get(var)
    print(f"{var}: {value if value else 'NOT FOUND'}")

# --- Save to .bashrc (no dictionary needed!) ---
bashrc_path = os.path.expanduser("~/.bashrc")

# Check if .bashrc exists, create it if not
if not os.path.exists(bashrc_path):
    print(f"Creating {bashrc_path}...")
    with open(bashrc_path, "w") as f:
        f.write("# Created by Verily setup script\n")

# Now continue with reading/appending
# Simple approach: just append (will create duplicates if run multiple times)
with open(bashrc_path, "a") as f:
    f.write("\n# Verily Workbench variables\n")
    for var in ["GOOGLE_CLOUD_PROJECT", "WORKSPACE_BUCKET", 
                "WORKSPACE_TEMP_BUCKET", "WORKSPACE_CDR"]:
        value = os.environ.get(var)
        if value:
            f.write(f'export {var}="{value}"\n')

print(f"\n✅ Saved to {bashrc_path}")

for r in resources:
    print(r["id"], r["resourceType"])
    if r["resourceType"] == "GCS_BUCKET" and "workspace-bucket" in r["id"]:
        os.environ["WORKSPACE_BUCKET"] = f"gs://{r['bucketName']}"
    elif r["resourceType"] == "GCS_BUCKET" and "temporary" in r["id"]:
        os.environ["WORKSPACE_TEMP_BUCKET"] = f"gs://{r['bucketName']}"

### Step 8. Reload the variables from `~/.bashrc`

In [ ]:
import os

# In your notebook, just run this:
with open(os.path.expanduser("~/.bashrc"), 'r') as f:
    for line in f:
        if line.strip().startswith('export '):
            parts = line.strip().replace('export ', '').split('=', 1)
            if len(parts) == 2:
                var_name = parts[0].strip()
                var_value = parts[1].strip().strip("'\"")
                
                # SKIP PATH completely!
                if var_name == 'PATH':
                    continue  # Skip this line
                    
                os.environ[var_name] = var_value


# %run /home/jupyter/Setting_Env_Variables_p2.ipynb

### Step 9. Write the R file every bootcamp notebook loads

Writes `/home/jupyter/Setting_Env_Variables_p2.R`. Each R notebook runs `source()` on this file in its setup cell.

In [ ]:
import os

# Define the variables to capture from your Notebook VM
vars_to_capture = [
    "WORKSPACE_CDR", 
    "GOOGLE_CLOUD_PROJECT", 
    "WORKSPACE_BUCKET", 
    "WORKSPACE_TEMP_BUCKET",
    "EXPORT_BUCKET" # Adding the 5th variable
]

r_file_path = "/home/jupyter/Setting_Env_Variables_p2.R"

with open(r_file_path, "w") as f:
    f.write("# --- AUTO-GENERATED ENVIRONMENT SETUP ---\n\n")
    
    # Part 1: Write the Sys.setenv lines
    for var in vars_to_capture:
        # Get value from environment
        value = os.environ.get(var)
        
        # Logic: If EXPORT_BUCKET is empty, use WORKSPACE_BUCKET value
        if var == "EXPORT_BUCKET" and not value:
            value = os.environ.get("WORKSPACE_BUCKET", "NOT_FOUND")
            
        # Fallback if the variable doesn't exist at all
        if not value:
            value = "NOT_FOUND"
            
        f.write(f'Sys.setenv({var} = "{value}")\n')
    
    # Part 2: Verification Block
    f.write("\n# --- VERIFICATION BLOCK ---\n")
    
    # Format the list for R: "VAR1", "VAR2", etc.
    r_list_str = ", ".join([f'"{v}"' for v in vars_to_capture])
    
    verification_code = f"""
vars_to_check <- c({r_list_str})

cat("\\n🔍 Current Workspace Variables:\\n")
for (v in vars_to_check) {{
  value <- Sys.getenv(v)
  # Prints the name (padded to 22 chars) and the value
  cat(sprintf("  %-22s : %s\\n", v, value))
}}
message("\\n✅ Environment Loaded Successfully.")
"""
    f.write(verification_code)

print(f"Done! {r_file_path} now contains {len(vars_to_capture)} variables.")

### Done

Your R notebooks will now load these variables with:

```r
source("/home/jupyter/Setting_Env_Variables_p2.R")
```

Run the cell below to see the file. Every line should show a value, not `NOT_FOUND`.

In [ ]:
print(open("/home/jupyter/Setting_Env_Variables_p2.R").read())